# Coin Change: Counting Permutations (Ordered Ways)

## Problem Statement
Given a set of coin denominations and a target `amount`, count the number of ways to make that amount, where **the order of coins matters**.

For example, with coins `{1, 2}` and amount `3`:
- `1+1+1`, `1+2`, and `2+1` are all counted, so `1+2` and `2+1` are **different** ways.

This is the *permutation* version of the coin change problem. In the *combination* version, `1+2` and `2+1` would count as one way.

## Approach: Bottom-Up Dynamic Programming
Let `dp[i]` be the number of ordered sequences of coins that sum to exactly `i`.

- **Base case:** `dp[0] = 1`, since there is one way to make 0 (use no coins).
- **Recurrence:** `dp[i] = Σ dp[i - coin]` for every `coin` with `coin <= i`.

The recurrence works because the **last coin** in a sequence can be any denomination. For each choice of last coin, the remaining amount `i - coin` can be formed in `dp[i - coin]` ordered ways.

## Step 1: Import Libraries

We import **NumPy**, used here for two things:
1. `np.zeros(...)` creates the DP table.
2. `np.array(...)` stores the coin denominations.

Plain Python lists would also work. NumPy is only a convenience.

In [4]:
import numpy as np

## Step 2: Define `coin_change_permutations(coins, amount)`

**Parameters**
- `coins`: the available coin denominations (assumed positive, unlimited supply of each).
- `amount`: the target sum.

**Returns:** the number of ordered ways to form `amount`.

### Line-by-line walkthrough
| Code | Explanation |
|------|-------------|
| `dp = np.zeros(amount+1)` | Creates a table of size `amount + 1`, filled with zeros. Index `i` stores the number of ways to make `i`. |
| `dp[0] = 1` | Base case: there is exactly one way to make 0, which is the empty selection. |
| `for i in range(1, amount+1)` | **Outer loop** over every sub-amount from 1 to `amount`. |
| `for coin in coins` | **Inner loop** that tries every coin as the *last* coin placed. |
| `if i >= coin` | Skips coins larger than the current sub-amount. |
| `dp[i] += dp[i-coin]` | Adds the number of ways to make the remaining amount `i - coin`. |
| `return dp[amount]` | The answer is the value at the target amount. |

### Why this counts permutations
The **amount loop is outside** and the **coin loop is inside**. For each `i`, every coin is considered as the final one in the sequence, so different orderings such as `[1,3]` and `[3,1]` are counted separately.

> **Contrast:** swapping the loops (coins outside, amounts inside) would count **combinations** instead.

### Note on data type
`np.zeros` creates a `float64` array by default, so the result is a float (e.g. `61.0`). Using `np.zeros(amount+1, dtype=int)` would return an integer and avoid float precision issues for very large counts.

In [5]:
def coin_change_permutations(coins, amount):
    dp = np.zeros(amount+1)
    dp[0]  =  1
    for i in range(1,amount+1):
        for coin in coins:
            if(i>=coin):
                dp[i] += dp[i-coin]
    return dp[amount]

## Step 3: Run the Function on a Sample Input

| Variable | Value | Meaning |
|----------|-------|---------|
| `coins` | `[6, 8, 5, 3, 1]` | Available denominations (order in the array doesn't affect the result) |
| `amount` | `10` | Target sum |

We call the function and print the result with an f-string. The coins array is intentionally unsorted to show that the algorithm doesn't need sorted input.

In [6]:
coins = np.array([6,8,5,3,1])
amount = 10

print(f"Number of Ways: {coin_change_permutations(coins,amount)}")

Number of Ways: 61.0


## Result

```
Number of Ways: 61.0
```

There are **61 ordered ways** to make `10` from the coins `{1, 3, 5, 6, 8}`.

### Dry-run of the DP table
Coins ≤ `i` are used at each step (sorted here for readability):

| `i` | Coins used | Calculation | `dp[i]` |
|-----|-----------|-------------|---------|
| 0 | (base case) | n/a | 1 |
| 1 | 1 | dp[0] | 1 |
| 2 | 1 | dp[1] | 1 |
| 3 | 1, 3 | dp[2] + dp[0] | 2 |
| 4 | 1, 3 | dp[3] + dp[1] | 3 |
| 5 | 1, 3, 5 | dp[4] + dp[2] + dp[0] | 5 |
| 6 | 1, 3, 5, 6 | dp[5] + dp[3] + dp[1] + dp[0] | 9 |
| 7 | 1, 3, 5, 6 | dp[6] + dp[4] + dp[2] + dp[1] | 14 |
| 8 | 1, 3, 5, 6, 8 | dp[7] + dp[5] + dp[3] + dp[2] + dp[0] | 23 |
| 9 | 1, 3, 5, 6, 8 | dp[8] + dp[6] + dp[4] + dp[3] + dp[1] | 38 |
| 10 | 1, 3, 5, 6, 8 | dp[9] + dp[7] + dp[5] + dp[4] + dp[2] | **61** |

The final value, `dp[10] = 61`, matches the program output.

## Complexity Analysis

Let **`n`** be the number of coin denominations and **`A`** the target `amount`.

### Time Complexity: **O(A × n)**
- The outer loop runs `A` times.
- For each iteration, the inner loop checks all `n` coins.
- Each check and update is O(1).
- For this example, that is `10 × 5 = 50` basic operations.

### Space Complexity: **O(A)**
- The only significant extra memory is the `dp` array of size `A + 1`.
- The coin list is the input, so it isn't counted as extra space.
- Only the last `max(coin)` entries are ever read, so memory could be reduced further with a circular buffer, but O(A) is the standard bound.

## Key Takeaways
- Loop order decides the problem: **amount outside, coins inside** counts *permutations*; **coins outside, amount inside** counts *combinations*.
- Use `dtype=int` (or Python ints) when counts can get large, to avoid floating-point precision loss.
- Assumes all coins are positive. A zero or negative coin would break the recurrence.